# Feature Engineering with NeuroKit2

This notebook uses a two-stage approach so feature engineering does not require running NeuroKit on all 528k windows immediately.

1. Run cheap NumPy QC on every complete 5-second PPG/ABP window.
2. Select a smaller set of windows for NeuroKit processing.
3. Merge NeuroKit-derived features back onto the full raw-QC table.

Raw `.mat` files are not overwritten. Outputs are saved under `data/features/neurokit_windows/`. The ABP columns are provisional targets only; they are not used as PPG input features.

In [9]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd

FS = 125
WINDOW_SIZE = 5 * FS
PART_FILES = [f"Part_{number}.mat" for number in range(1, 5)]

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").is_dir():
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
FEATURE_DIR = ROOT / "data" / "features" / "neurokit_windows"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

missing_files = [name for name in PART_FILES if not (RAW_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing raw files in {RAW_DIR}: {', '.join(missing_files)}. "
        "See data/README.md for setup instructions."
    )

print(f"NeuroKit2 version: {nk.__version__}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Feature output directory: {FEATURE_DIR}")
print("If imports are missing, install with: python -m pip install neurokit2 pandas matplotlib")

NeuroKit2 version: 0.2.13
Raw data directory: /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/raw
Feature output directory: /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows
If imports are missing, install with: python -m pip install neurokit2 pandas matplotlib


## Helper Functions

`raw_window_qc_row` is fast and runs on every window. `extract_neurokit_window_features` is slower and runs only on selected windows.

In [10]:
def longest_true_run(mask):
    """Return the longest consecutive True run in a 1D boolean array."""
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return 0
    padded = np.concatenate(([False], mask, [False]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    return int((changes[1::2] - changes[::2]).max())


def finite_nanmedian(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    return float(np.median(finite)) if len(finite) else np.nan


def raw_window_qc_row(ppg, abp, part, part_number, record_index, window_index, global_window_index):
    return {
        "part": part,
        "part_number": int(part_number),
        "record_index": int(record_index),
        "window_index": int(window_index),
        "global_window_index": int(global_window_index),
        "sbp_window_max_target": float(np.max(abp)),
        "dbp_window_min_target": float(np.min(abp)),
        "abp_window_range_qc": float(np.ptp(abp)),
        "ppg_nonfinite_count_qc": int((~np.isfinite(ppg)).sum()),
        "abp_nonfinite_count_qc": int((~np.isfinite(abp)).sum()),
        "ppg_zero_count_qc": int(np.count_nonzero(ppg == 0)),
        "ppg_zero_fraction_qc": float(np.mean(ppg == 0)),
        "ppg_longest_zero_run_qc": longest_true_run(ppg == 0),
        "ppg_raw_min_qc": float(np.min(ppg)),
        "ppg_raw_max_qc": float(np.max(ppg)),
        "ppg_raw_range_qc": float(np.ptp(ppg)),
        "ppg_raw_std_qc": float(np.std(ppg)),
    }


def extract_neurokit_window_features(ppg, abp):
    row = {
        "processing_status": "failed",
        "processing_error": "",
    }

    try:
        if not np.isfinite(ppg).all():
            raise ValueError("PPG contains non-finite values")
        if not np.isfinite(abp).all():
            raise ValueError("ABP contains non-finite values")

        signals, info = nk.ppg_process(ppg, sampling_rate=FS)
        clean = signals["PPG_Clean"].to_numpy(dtype=float)
        peak_mask = signals["PPG_Peaks"].to_numpy(dtype=int) > 0
        peak_indices = np.flatnonzero(peak_mask)
        peak_intervals = np.diff(peak_indices) / FS
        pulse_rates = 60.0 / peak_intervals if len(peak_intervals) else np.array([])
        rate_series = signals["PPG_Rate"].to_numpy(dtype=float)
        quality_series = signals["PPG_Quality"].to_numpy(dtype=float)

        pulse_amplitudes = []
        for previous_peak, current_peak in zip(peak_indices[:-1], peak_indices[1:]):
            segment = clean[previous_peak:current_peak + 1]
            if len(segment):
                pulse_amplitudes.append(float(clean[current_peak] - np.min(segment)))

        row.update(
            {
                "ppg_clean_mean_feature": float(np.mean(clean)),
                "ppg_clean_std_feature": float(np.std(clean)),
                "ppg_clean_range_feature": float(np.ptp(clean)),
                "detected_peak_count_qc": int(len(peak_indices)),
                "median_interpeak_interval_s_feature": float(np.median(peak_intervals)) if len(peak_intervals) else np.nan,
                "min_interpeak_interval_s_qc": float(np.min(peak_intervals)) if len(peak_intervals) else np.nan,
                "max_interpeak_interval_s_qc": float(np.max(peak_intervals)) if len(peak_intervals) else np.nan,
                "median_rate_from_peaks_bpm_feature": float(np.median(pulse_rates)) if len(pulse_rates) else np.nan,
                "min_rate_from_peaks_bpm_qc": float(np.min(pulse_rates)) if len(pulse_rates) else np.nan,
                "max_rate_from_peaks_bpm_qc": float(np.max(pulse_rates)) if len(pulse_rates) else np.nan,
                "median_neurokit_rate_bpm_qc": finite_nanmedian(rate_series),
                "median_pulse_amplitude_feature": float(np.median(pulse_amplitudes)) if pulse_amplitudes else np.nan,
                "median_neurokit_quality_qc": finite_nanmedian(quality_series),
                "processing_status": "ok",
            }
        )
    except Exception as error:
        row["processing_error"] = f"{type(error).__name__}: {error}"

    return row

## Stage 1: Fast Raw QC for the Whole Dataset

This pass is cheap because it uses vectorized NumPy summaries, not NeuroKit. It scans every complete 625-sample window and writes one raw-QC CSV per part. Re-running is resumable because existing part files are skipped.

In [11]:
SKIP_EXISTING_RAW_QC_FILES = True
PROGRESS_EVERY_RECORDS = 250

raw_qc_outputs = []
global_window_offset = 0

for part_number, filename in enumerate(PART_FILES, start=1):
    output_path = FEATURE_DIR / f"{filename.replace('.mat', '')}_raw_window_qc.csv"
    raw_qc_outputs.append(output_path)

    if SKIP_EXISTING_RAW_QC_FILES and output_path.exists():
        existing = pd.read_csv(output_path, usecols=["global_window_index"])
        if len(existing):
            global_window_offset = max(global_window_offset, int(existing["global_window_index"].max()) + 1)
        print(f"Skipping existing {output_path} ({len(existing):,} rows)")
        continue

    rows = []
    with h5py.File(RAW_DIR / filename, "r") as source:
        references = source[f"Part_{part_number}"]
        print(f"Raw QC for {filename}: {references.shape[0]:,} records")

        for record_index in range(references.shape[0]):
            record = source[references[record_index, 0]][()]
            if record.ndim != 2 or record.shape[1] != 3:
                raise ValueError(f"Unexpected shape in {filename}, record {record_index}: {record.shape}")

            window_count = record.shape[0] // WINDOW_SIZE
            if not window_count:
                continue

            windows = record[: window_count * WINDOW_SIZE].reshape(window_count, WINDOW_SIZE, 3)
            ppg_windows = windows[:, :, 0].astype(float, copy=False)
            abp_windows = windows[:, :, 1].astype(float, copy=False)

            for window_index, (ppg, abp) in enumerate(zip(ppg_windows, abp_windows)):
                rows.append(
                    raw_window_qc_row(
                        ppg=ppg,
                        abp=abp,
                        part=filename,
                        part_number=part_number,
                        record_index=record_index,
                        window_index=window_index,
                        global_window_index=global_window_offset + len(rows),
                    )
                )

            if record_index % PROGRESS_EVERY_RECORDS == 0:
                print(f"  {filename}: raw-QC record {record_index:,}/{references.shape[0]:,}; windows so far {len(rows):,}")

    part_raw_qc = pd.DataFrame(rows)
    part_raw_qc.to_csv(output_path, index=False)
    global_window_offset += len(part_raw_qc)
    print(f"Saved {output_path} with {len(part_raw_qc):,} windows")

Raw QC for Part_1.mat: 3,000 records
  Part_1.mat: raw-QC record 0/3,000; windows so far 97
  Part_1.mat: raw-QC record 250/3,000; windows so far 15,168
  Part_1.mat: raw-QC record 500/3,000; windows so far 32,021
  Part_1.mat: raw-QC record 750/3,000; windows so far 44,476
  Part_1.mat: raw-QC record 1,000/3,000; windows so far 50,886
  Part_1.mat: raw-QC record 1,250/3,000; windows so far 62,648
  Part_1.mat: raw-QC record 1,500/3,000; windows so far 72,999
  Part_1.mat: raw-QC record 1,750/3,000; windows so far 84,461
  Part_1.mat: raw-QC record 2,000/3,000; windows so far 95,444
  Part_1.mat: raw-QC record 2,250/3,000; windows so far 106,917
  Part_1.mat: raw-QC record 2,500/3,000; windows so far 118,507
  Part_1.mat: raw-QC record 2,750/3,000; windows so far 125,596
Saved /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows/Part_1_raw_window_qc.csv with 131,163 windows
Raw QC for Part_2.mat: 3,000 records
  Part_2.mat:

## Stage 2: Flag Cheap Raw Outliers

These flags are available for the whole dataset before any NeuroKit call. They are also used to decide which windows need the expensive NeuroKit pass.

In [12]:
missing_raw_qc = [path for path in raw_qc_outputs if not path.exists()]
if missing_raw_qc:
    raise FileNotFoundError(f"Missing raw QC files: {missing_raw_qc}")

raw_features = pd.concat((pd.read_csv(path) for path in raw_qc_outputs), ignore_index=True)
raw_features = raw_features.sort_values(["part_number", "record_index", "window_index"]).reset_index(drop=True)

raw_low_range_threshold = raw_features["ppg_raw_range_qc"].quantile(0.01)
raw_high_range_threshold = raw_features["ppg_raw_range_qc"].quantile(0.99)
raw_low_std_threshold = raw_features["ppg_raw_std_qc"].quantile(0.01)
raw_high_std_threshold = raw_features["ppg_raw_std_qc"].quantile(0.99)

raw_features["raw_nonfinite_outlier_qc"] = (raw_features["ppg_nonfinite_count_qc"] > 0) | (raw_features["abp_nonfinite_count_qc"] > 0)
raw_features["raw_zero_ppg_outlier_qc"] = raw_features["ppg_zero_count_qc"] > 0
raw_features["raw_long_zero_run_outlier_qc"] = raw_features["ppg_longest_zero_run_qc"] >= 32
raw_features["raw_low_ppg_range_outlier_qc"] = raw_features["ppg_raw_range_qc"] <= raw_low_range_threshold
raw_features["raw_high_ppg_range_outlier_qc"] = raw_features["ppg_raw_range_qc"] >= raw_high_range_threshold
raw_features["raw_low_ppg_std_outlier_qc"] = raw_features["ppg_raw_std_qc"] <= raw_low_std_threshold
raw_features["raw_high_ppg_std_outlier_qc"] = raw_features["ppg_raw_std_qc"] >= raw_high_std_threshold
raw_features["raw_abp_label_outlier_qc"] = (
    raw_features["sbp_window_max_target"].gt(250)
    | raw_features["dbp_window_min_target"].lt(40)
    | raw_features["abp_window_range_qc"].lt(5)
)

raw_outlier_columns = [column for column in raw_features.columns if column.startswith("raw_") and column.endswith("_outlier_qc")]
raw_features["any_raw_outlier_qc"] = raw_features[raw_outlier_columns].any(axis=1)
raw_features["raw_outlier_reason_qc"] = raw_features[raw_outlier_columns].apply(
    lambda row: ";".join(column.replace("raw_", "").replace("_outlier_qc", "") for column, flagged in row.items() if flagged),
    axis=1,
)

raw_qc_all_path = FEATURE_DIR / "raw_qc_all_windows.csv"
raw_features.to_csv(raw_qc_all_path, index=False)

print(f"Raw QC windows: {len(raw_features):,}")
print(f"Raw outlier windows: {int(raw_features['any_raw_outlier_qc'].sum()):,}")
print(f"Saved {raw_qc_all_path}")
display(raw_features[raw_outlier_columns + ["any_raw_outlier_qc"]].sum().sort_values(ascending=False).to_frame("window_count"))

Raw QC windows: 528,828
Raw outlier windows: 17,277
Saved /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows/raw_qc_all_windows.csv


,window_count
any_raw_outlier_qc,17277
raw_low_ppg_range_outlier_qc,5425
raw_zero_ppg_outlier_qc,5407
raw_high_ppg_range_outlier_qc,5293
raw_low_ppg_std_outlier_qc,5289
raw_high_ppg_std_outlier_qc,5289
raw_nonfinite_outlier_qc,0
raw_long_zero_run_outlier_qc,0
raw_abp_label_outlier_qc,0


## Stage 3: Select Windows for NeuroKit

This is the main compute-control switch. Start with `qc_pass_sample` or `sample`, then increase `NEUROKIT_SAMPLE_N` only if needed. Use `all` only when you are ready for the full expensive pass.

In [13]:
NEUROKIT_SELECTION_MODE = "qc_pass_sample"
NEUROKIT_SAMPLE_N = 20_000
RANDOM_SEED = 2026

valid_modes = {"all", "qc_pass", "raw_outliers", "sample", "qc_pass_sample"}
if NEUROKIT_SELECTION_MODE not in valid_modes:
    raise ValueError(f"NEUROKIT_SELECTION_MODE must be one of {sorted(valid_modes)}")

if NEUROKIT_SELECTION_MODE == "all":
    selected_windows = raw_features.copy()
elif NEUROKIT_SELECTION_MODE == "qc_pass":
    selected_windows = raw_features.loc[~raw_features["any_raw_outlier_qc"]].copy()
elif NEUROKIT_SELECTION_MODE == "raw_outliers":
    selected_windows = raw_features.loc[raw_features["any_raw_outlier_qc"]].copy()
elif NEUROKIT_SELECTION_MODE == "sample":
    selected_windows = raw_features.sample(n=min(NEUROKIT_SAMPLE_N, len(raw_features)), random_state=RANDOM_SEED).copy()
elif NEUROKIT_SELECTION_MODE == "qc_pass_sample":
    pool = raw_features.loc[~raw_features["any_raw_outlier_qc"]]
    selected_windows = pool.sample(n=min(NEUROKIT_SAMPLE_N, len(pool)), random_state=RANDOM_SEED).copy()

selected_windows = selected_windows.sort_values(["part_number", "record_index", "window_index"]).reset_index(drop=True)
selection_path = FEATURE_DIR / f"neurokit_selection_{NEUROKIT_SELECTION_MODE}.csv"
selected_windows.to_csv(selection_path, index=False)

selected_key_columns = ["part", "record_index", "window_index"]
selected_key_set = set(map(tuple, selected_windows[selected_key_columns].to_numpy()))

print(f"NeuroKit selection mode: {NEUROKIT_SELECTION_MODE}")
print(f"Selected windows for NeuroKit: {len(selected_windows):,} / {len(raw_features):,}")
print(f"Saved {selection_path}")
display(selected_windows.head())

NeuroKit selection mode: qc_pass_sample
Selected windows for NeuroKit: 20,000 / 528,828
Saved /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows/neurokit_selection_qc_pass_sample.csv


,part,part_number,record_index,window_index,global_window_index,sbp_window_max_target,dbp_window_min_target,abp_window_range_qc,ppg_nonfinite_count_qc,abp_nonfinite_count_qc,...,raw_nonfinite_outlier_qc,raw_zero_ppg_outlier_qc,raw_long_zero_run_outlier_qc,raw_low_ppg_range_outlier_qc,raw_high_ppg_range_outlier_qc,raw_low_ppg_std_outlier_qc,raw_high_ppg_std_outlier_qc,raw_abp_label_outlier_qc,any_raw_outlier_qc,raw_outlier_reason_qc
0,Part_1.mat,1,0,12,12,130.316070,67.502552,62.813518,0,0,...,False,False,False,False,False,False,False,False,False,
1,Part_1.mat,1,0,49,49,141.257150,75.317609,65.939541,0,0,...,False,False,False,False,False,False,False,False,False,
2,Part_1.mat,1,1,5,102,134.077066,63.155427,70.921639,0,0,...,False,False,False,False,False,False,False,False,False,
3,Part_1.mat,1,1,66,163,132.172146,70.726263,61.445883,0,0,...,False,False,False,False,False,False,False,False,False,
4,Part_1.mat,1,2,48,242,140.426800,80.983525,59.443275,0,0,...,False,False,False,False,False,False,False,False,False,


## Stage 4: Run NeuroKit on Selected Windows Only

This cell reopens the raw files, but it only calls `nk.ppg_process` when a window is in `selected_windows`. Results are saved per part and can be resumed.

In [14]:
SKIP_EXISTING_NEUROKIT_FILES = True
PROGRESS_EVERY_RECORDS = 250

neurokit_outputs = []

for part_number, filename in enumerate(PART_FILES, start=1):
    output_path = FEATURE_DIR / f"{filename.replace('.mat', '')}_neurokit_{NEUROKIT_SELECTION_MODE}.csv"
    neurokit_outputs.append(output_path)

    if SKIP_EXISTING_NEUROKIT_FILES and output_path.exists():
        existing = pd.read_csv(output_path, usecols=["part", "record_index", "window_index"])
        print(f"Skipping existing {output_path} ({len(existing):,} rows)")
        continue

    selected_for_part = selected_windows.loc[selected_windows["part"].eq(filename)]
    if selected_for_part.empty:
        pd.DataFrame(columns=selected_key_columns + ["processing_status", "processing_error"]).to_csv(output_path, index=False)
        print(f"No selected windows for {filename}; wrote empty {output_path}")
        continue

    part_key_set = set(map(tuple, selected_for_part[selected_key_columns].to_numpy()))
    rows = []
    with h5py.File(RAW_DIR / filename, "r") as source:
        references = source[f"Part_{part_number}"]
        print(f"NeuroKit for {filename}: {len(selected_for_part):,} selected windows")

        for record_index in range(references.shape[0]):
            candidate_keys = [key for key in part_key_set if key[1] == record_index]
            if not candidate_keys:
                continue

            record = source[references[record_index, 0]][()]
            window_count = record.shape[0] // WINDOW_SIZE
            windows = record[: window_count * WINDOW_SIZE].reshape(window_count, WINDOW_SIZE, 3)

            for _, _, window_index in candidate_keys:
                window = windows[int(window_index)]
                ppg = window[:, 0].astype(float, copy=True)
                abp = window[:, 1].astype(float, copy=True)
                row = {"part": filename, "record_index": int(record_index), "window_index": int(window_index)}
                row.update(extract_neurokit_window_features(ppg, abp))
                rows.append(row)

            if record_index % PROGRESS_EVERY_RECORDS == 0:
                print(f"  {filename}: NeuroKit record {record_index:,}/{references.shape[0]:,}; processed {len(rows):,}/{len(selected_for_part):,}")

    part_neurokit = pd.DataFrame(rows)
    part_neurokit.to_csv(output_path, index=False)
    print(f"Saved {output_path} with {len(part_neurokit):,} NeuroKit rows")

NeuroKit for Part_1.mat: 4,866 selected windows
  Part_1.mat: NeuroKit record 0/3,000; processed 2/4,866
  Part_1.mat: NeuroKit record 250/3,000; processed 546/4,866
  Part_1.mat: NeuroKit record 500/3,000; processed 1,212/4,866
  Part_1.mat: NeuroKit record 750/3,000; processed 1,649/4,866
  Part_1.mat: NeuroKit record 1,750/3,000; processed 3,205/4,866


/Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/.venv/lib/python3.13/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


  Part_1.mat: NeuroKit record 2,000/3,000; processed 3,606/4,866
  Part_1.mat: NeuroKit record 2,250/3,000; processed 3,986/4,866
Saved /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows/Part_1_neurokit_qc_pass_sample.csv with 4,866 NeuroKit rows
NeuroKit for Part_2.mat: 5,702 selected windows
  Part_2.mat: NeuroKit record 0/3,000; processed 1/5,702
  Part_2.mat: NeuroKit record 250/3,000; processed 414/5,702


/Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/.venv/lib/python3.13/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


  Part_2.mat: NeuroKit record 750/3,000; processed 1,308/5,702
  Part_2.mat: NeuroKit record 1,000/3,000; processed 1,881/5,702
  Part_2.mat: NeuroKit record 1,500/3,000; processed 2,943/5,702
  Part_2.mat: NeuroKit record 1,750/3,000; processed 3,313/5,702


/Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/.venv/lib/python3.13/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


  Part_2.mat: NeuroKit record 2,000/3,000; processed 3,796/5,702
  Part_2.mat: NeuroKit record 2,250/3,000; processed 4,404/5,702
  Part_2.mat: NeuroKit record 2,500/3,000; processed 4,894/5,702
  Part_2.mat: NeuroKit record 2,750/3,000; processed 5,288/5,702
Saved /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows/Part_2_neurokit_qc_pass_sample.csv with 5,702 NeuroKit rows
NeuroKit for Part_3.mat: 4,301 selected windows
  Part_3.mat: NeuroKit record 250/3,000; processed 164/4,301
  Part_3.mat: NeuroKit record 500/3,000; processed 327/4,301
  Part_3.mat: NeuroKit record 1,250/3,000; processed 1,509/4,301
  Part_3.mat: NeuroKit record 1,750/3,000; processed 2,028/4,301
  Part_3.mat: NeuroKit record 2,000/3,000; processed 2,384/4,301
  Part_3.mat: NeuroKit record 2,500/3,000; processed 3,501/4,301
Saved /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows/Part_3

/Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/.venv/lib/python3.13/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/.venv/lib/python3.13/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/.venv/lib/python3.13/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


Saved /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows/Part_4_neurokit_qc_pass_sample.csv with 5,131 NeuroKit rows


## Stage 5: Merge Raw QC and NeuroKit Results

Rows not selected for NeuroKit keep their raw QC and receive `processing_status = "not_run"`. This is intentional: you can still find raw outliers across the whole dataset without paying for NeuroKit everywhere.

In [15]:
missing_neurokit_outputs = [path for path in neurokit_outputs if not path.exists()]
if missing_neurokit_outputs:
    raise FileNotFoundError(f"Missing NeuroKit files: {missing_neurokit_outputs}")

neurokit_frames = [pd.read_csv(path) for path in neurokit_outputs if path.stat().st_size > 0]
neurokit_features = pd.concat(neurokit_frames, ignore_index=True) if neurokit_frames else pd.DataFrame(columns=selected_key_columns)

features = raw_features.merge(
    neurokit_features,
    on=selected_key_columns,
    how="left",
    validate="one_to_one",
)
features["processing_status"] = features["processing_status"].fillna("not_run")
features["processing_error"] = features["processing_error"].fillna("")
features["neurokit_selected_qc"] = features["processing_status"].ne("not_run")

combined_path = FEATURE_DIR / f"features_raw_qc_plus_neurokit_{NEUROKIT_SELECTION_MODE}.csv"
features.to_csv(combined_path, index=False)

print(f"Combined table: {features.shape[0]:,} windows x {features.shape[1]:,} columns")
print(f"NeuroKit status counts:")
display(features["processing_status"].value_counts(dropna=False))
print(f"Saved {combined_path}")
display(features.head())

Combined table: 528,828 windows x 43 columns
NeuroKit status counts:


processing_status
not_run    508828
ok          19994
failed          6
Name: count, dtype: int64

Saved /Users/sarahto/Desktop/Coding/BTT/Microsoft-1D-educational-blood-pressure-estimation/data/features/neurokit_windows/features_raw_qc_plus_neurokit_qc_pass_sample.csv


,part,part_number,record_index,window_index,global_window_index,sbp_window_max_target,dbp_window_min_target,abp_window_range_qc,ppg_nonfinite_count_qc,abp_nonfinite_count_qc,...,median_interpeak_interval_s_feature,min_interpeak_interval_s_qc,max_interpeak_interval_s_qc,median_rate_from_peaks_bpm_feature,min_rate_from_peaks_bpm_qc,max_rate_from_peaks_bpm_qc,median_neurokit_rate_bpm_qc,median_pulse_amplitude_feature,median_neurokit_quality_qc,neurokit_selected_qc
0,Part_1.mat,1,0,0,0,125.187439,65.597632,59.589807,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,Part_1.mat,1,0,1,1,127.434268,67.307176,60.127092,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,Part_1.mat,1,0,2,2,127.190048,68.381746,58.808302,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,Part_1.mat,1,0,3,3,124.308245,66.037229,58.271016,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,Part_1.mat,1,0,4,4,128.508838,67.551396,60.957442,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


## Stage 6: Final Outlier Flags

Final flags combine whole-dataset raw QC and NeuroKit-derived QC where NeuroKit was run. A `not_run` window is not automatically a bad window; it simply lacks NeuroKit-derived evidence.

In [ ]:
features["failed_processing_outlier_qc"] = features["processing_status"].eq("failed")
features["low_neurokit_quality_outlier_qc"] = features["neurokit_selected_qc"] & features["median_neurokit_quality_qc"].lt(0.5)
features["peak_count_outlier_qc"] = features["neurokit_selected_qc"] & (
    features["detected_peak_count_qc"].lt(3) | features["detected_peak_count_qc"].gt(12)
)
features["rate_outlier_qc"] = features["neurokit_selected_qc"] & (
    features["median_rate_from_peaks_bpm_feature"].lt(40) | features["median_rate_from_peaks_bpm_feature"].gt(180)
)

neurokit_outlier_columns = [
    "failed_processing_outlier_qc",
    "low_neurokit_quality_outlier_qc",
    "peak_count_outlier_qc",
    "rate_outlier_qc",
]
outlier_columns = raw_outlier_columns + neurokit_outlier_columns
features["any_outlier_qc"] = features[outlier_columns].any(axis=1)
features["outlier_reason_qc"] = features[outlier_columns].apply(
    lambda row: ";".join(column.replace("_outlier_qc", "") for column, flagged in row.items() if flagged),
    axis=1,
)

features_with_flags_path = FEATURE_DIR / f"features_with_outlier_flags_{NEUROKIT_SELECTION_MODE}.csv"
outliers_path = FEATURE_DIR / f"outlier_windows_{NEUROKIT_SELECTION_MODE}.csv"
features.to_csv(features_with_flags_path, index=False)
features.loc[features["any_outlier_qc"]].to_csv(outliers_path, index=False)

print(f"Any outlier windows: {int(features['any_outlier_qc'].sum()):,} / {len(features):,}")
print(f"Saved {features_with_flags_path}")
print(f"Saved {outliers_path}")

## Outlier Summary

In [ ]:
outlier_summary = (
    features[outlier_columns + ["any_outlier_qc"]]
    .sum()
    .rename("window_count")
    .to_frame()
)
outlier_summary["window_percent"] = 100 * outlier_summary["window_count"] / len(features)
outlier_summary = outlier_summary.sort_values("window_count", ascending=False)

summary_path = FEATURE_DIR / f"outlier_summary_{NEUROKIT_SELECTION_MODE}.csv"
outlier_summary.to_csv(summary_path)

display(outlier_summary)
display(
    features.loc[features["any_outlier_qc"], [
        "part", "record_index", "window_index", "global_window_index",
        "processing_status", "neurokit_selected_qc", "outlier_reason_qc",
        "ppg_zero_count_qc", "ppg_raw_range_qc",
        "detected_peak_count_qc", "median_rate_from_peaks_bpm_feature",
        "median_neurokit_quality_qc", "sbp_window_max_target", "dbp_window_min_target",
    ]].head(20)
)
print(f"Saved {summary_path}")

## Optional Visual Inspection of Flagged Windows

The flags are only candidates until inspected. This helper reloads selected raw windows, reruns NeuroKit for plotting, and overlays raw PPG, cleaned PPG, and detected peaks.

In [ ]:
def load_raw_window(part, record_index, window_index):
    part_number = int(str(part).split("_")[1].split(".")[0])
    with h5py.File(RAW_DIR / part, "r") as source:
        references = source[f"Part_{part_number}"]
        record = source[references[int(record_index), 0]][()]
    start = int(window_index) * WINDOW_SIZE
    stop = start + WINDOW_SIZE
    window = record[start:stop]
    return window[:, 0].astype(float, copy=True), window[:, 1].astype(float, copy=True)


def plot_outlier_examples(rows, max_examples=6):
    rows = rows.head(max_examples)
    if rows.empty:
        print("No rows to plot.")
        return

    time_seconds = np.arange(WINDOW_SIZE) / FS
    fig, axes = plt.subplots(len(rows), 1, figsize=(12, 3.5 * len(rows)), squeeze=False)
    for axis, (_, row) in zip(axes[:, 0], rows.iterrows()):
        ppg, abp = load_raw_window(row["part"], row["record_index"], row["window_index"])
        axis.plot(time_seconds, ppg, alpha=0.55, linewidth=0.8, label="Raw PPG")
        try:
            signals, info = nk.ppg_process(ppg, sampling_rate=FS)
            clean = signals["PPG_Clean"].to_numpy(dtype=float)
            peaks = np.flatnonzero(signals["PPG_Peaks"].to_numpy(dtype=int) > 0)
            axis.plot(time_seconds, clean, linewidth=1.0, label="NeuroKit cleaned PPG")
            axis.scatter(peaks / FS, clean[peaks], s=20, color="tab:red", label="Detected peaks", zorder=3)
        except Exception as error:
            axis.text(0.02, 0.9, f"NeuroKit failed: {type(error).__name__}: {error}", transform=axis.transAxes)

        axis.set_title(
            f"{row['part']} record {int(row['record_index'])}, window {int(row['window_index'])} | "
            f"reasons: {row['outlier_reason_qc']}"
        )
        axis.set_xlabel("Time (seconds)")
        axis.set_ylabel("PPG (dataset units)")
        axis.grid(alpha=0.25)
        axis.legend(loc="upper right")
    fig.tight_layout()
    plt.show()


example_outliers = features.loc[features["any_outlier_qc"]].sample(
    n=min(6, int(features["any_outlier_qc"].sum())),
    random_state=2026,
)
plot_outlier_examples(example_outliers)

## Training Table

For model training, start from the flagged table and filter out review-failed windows. Keep the full table too, so exclusions are auditable.

In [ ]:
model_ready = features.loc[
    features["processing_status"].isin(["ok", "not_run"]) & ~features["any_outlier_qc"]
].copy()

model_ready_path = FEATURE_DIR / f"model_ready_windows_{NEUROKIT_SELECTION_MODE}.csv"
model_ready.to_csv(model_ready_path, index=False)

print(f"Model-ready windows after current QC filter: {len(model_ready):,} / {len(features):,}")
print(f"Saved {model_ready_path}")